# Cognivision Voice Intelligence
## Notebook 05: Model Evaluation, Metrics, and Error Analysis
This notebook evaluates the trained PyTorch 2D-CNN on the untouched test split, generating the confusion matrix, classification report, One-vs-Rest ROC/PR curves, and error analysis.

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

root_dir = Path("..").resolve()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from backend.app.services.evaluation_service import evaluation_service
from backend.app.utils.constants import EMOTIONS

### 1. Load Evaluation Results

In [ ]:
eval_report = evaluation_service.load_latest_eval_if_exists()
if eval_report is None:
    print("Running evaluation on test split...")
    eval_report = evaluation_service.evaluate_model(model_type="cnn")

m = eval_report['metrics']
print("=" * 50)
print(f"Overall Accuracy:        {m['accuracy']}%")
print(f"Macro F1-Score:          {m['macro_f1']}%")
print(f"Weighted F1-Score:       {m['weighted_f1']}%")
print(f"Balanced Accuracy:       {m['balanced_accuracy']}%")
print(f"Macro AUC:               {m['macro_auc']}")
print("=" * 50)

### 2. Per-Class Metrics Table

In [ ]:
per_class_df = pd.DataFrame(eval_report['per_class'])
per_class_df

### 3. Confusion Matrix Visualization

In [ ]:
cm = np.array(eval_report['confusion_matrix']['counts'])
cm_norm = np.array(eval_report['confusion_matrix']['normalized'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=EMOTIONS, yticklabels=EMOTIONS, ax=ax1)
ax1.set_title("Confusion Matrix (Counts)", fontweight='bold')
ax1.set_xlabel("Predicted Emotion")
ax1.set_ylabel("True Emotion")

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=EMOTIONS, yticklabels=EMOTIONS, ax=ax2)
ax2.set_title("Normalized Confusion Matrix", fontweight='bold')
ax2.set_xlabel("Predicted Emotion")
ax2.set_ylabel("True Emotion")

plt.tight_layout()
plt.show()

### 4. Error Analysis: Top Confused Pairs

In [ ]:
err_analysis = eval_report.get('error_analysis', {})
top_pairs = err_analysis.get('top_confused_pairs', [])

if top_pairs:
    pairs_df = pd.DataFrame(top_pairs)
    plt.figure(figsize=(9, 4))
    sns.barplot(data=pairs_df, x='pair', y='count', palette='rocket')
    plt.title("Top Confused Emotion Pairs", fontweight='bold')
    plt.xlabel("True -> Predicted Pair")
    plt.ylabel("Number of Errors")
    plt.xticks(rotation=30, ha='right')
    plt.grid(axis='y', alpha=0.3)
    plt.show()